Question2: Moderate (CO4) – Dataset14: Evaluate a polynomial using CUDA parallel threads. (Dataset14: HIGGS Dataset)

In [1]:
!pip install numba -q

import numpy as np
import math
import numba
from numba import cuda
import pandas as pd
import time

# 1. Download the HIGGS dataset (compressed) from UCI repository
import os
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00280/HIGGS.csv.gz"
file_name = "HIGGS.csv.gz"
if not os.path.exists(file_name):
    !wget -q {url}
    print("Dataset downloaded.")
else:
    print("Dataset already exists.")

Dataset downloaded.


In [2]:
# 2. Load a manageable chunk (first 1 million rows) to avoid memory issues
# The dataset has no header; 28 feature columns, last column is the label.
print("Loading data...")
chunk_size = 1_000_000
df = pd.read_csv(file_name, header=None, nrows=chunk_size)
# Use column 1 (index 1) as our feature vector, convert to float32
feature = df.iloc[:, 1].values.astype(np.float32)
N = len(feature)
print(f"Loaded {N} values from feature column 1.")

Loading data...
Loaded 1000000 values from feature column 1.


In [3]:
# 3. Define polynomial coefficients (example: 5th degree polynomial)
# f(x) = 2.5 - 3.1*x + 1.2*x^2 + 0.7*x^3 - 0.3*x^4 + 0.05*x^5
coeffs = np.array([2.5, -3.1, 1.2, 0.7, -0.3, 0.05], dtype=np.float32)
degree = len(coeffs) - 1

In [4]:
# 4. CUDA kernel for polynomial evaluation using Horner's rule
@cuda.jit
def poly_kernel(x, result, coeffs, n):
    """Each thread evaluates the polynomial at x[idx] and stores in result[idx]."""
    idx = cuda.grid(1)
    if idx < n:
        val = x[idx]
        # Horner's method: start from highest degree coefficient
        res = coeffs[degree]  # degree is known on host, pass as scalar? We'll use coeffs array.
        # But we need degree as a constant inside kernel. We'll do a loop from degree down to 0.
        # Since kernel can't access Python len, we pass degree as parameter.
        # Here I'm showing a simple loop using degree variable from host via closure.
        # In numba, we can access global constants? Better to pass degree as a scalar.
        # Simpler: for this student code, we'll hardcode degree = 5.
        # Actually we pass coeffs array and degree as a parameter; so modify kernel signature.
        pass

# I'll rewrite kernel properly with degree parameter.
@cuda.jit
def poly_kernel(x, result, coeffs, degree):
    idx = cuda.grid(1)
    if idx < len(x):
        # Horner's rule: (((c5*x + c4)*x + c3)*x + c2)*x + c1)*x + c0
        val = x[idx]
        res = coeffs[degree]
        for i in range(degree-1, -1, -1):
            res = res * val + coeffs[i]
        result[idx] = res

In [5]:
# 5. Transfer data to GPU and prepare result array
d_x = cuda.to_device(feature)
d_result = cuda.device_array_like(feature)
d_coeffs = cuda.to_device(coeffs)

In [6]:
# 6. Configure grid & blocks
threads_per_block = 256
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block


In [7]:
# 7. Run kernel and measure time
start = time.time()
poly_kernel[blocks_per_grid, threads_per_block](d_x, d_result, d_coeffs, degree)
cuda.synchronize()
gpu_time = time.time() - start

In [8]:
# 8. Copy result back to host
result_gpu = d_result.copy_to_host()

# 9. CPU reference computation for verification (first few elements)
cpu_poly = np.zeros(N, dtype=np.float32)
for i in range(N):
    val = feature[i]
    res = coeffs[degree]
    for j in range(degree-1, -1, -1):
        res = res * val + coeffs[j]
    cpu_poly[i] = res

In [9]:
# Compare a small subset
print("\nVerification (first 10 values):")
for i in range(10):
    print(f"x={feature[i]:.4f}, GPU={result_gpu[i]:.6f}, CPU={cpu_poly[i]:.6f}")


Verification (first 10 values):
x=0.8693, GPU=1.025334, CPU=1.025334
x=0.9075, GPU=1.025487, CPU=1.025487
x=0.7988, GPU=1.040312, CPU=1.040312
x=1.3444, GPU=1.441714, CPU=1.441714
x=1.1050, GPU=1.119303, CPU=1.119303
x=1.5958, GPU=2.025630, CPU=2.025630
x=0.4094, GPU=1.472187, CPU=1.472187
x=0.9339, GPU=1.028992, CPU=1.028992
x=1.4051, GPU=1.559796, CPU=1.559797
x=1.1766, GPU=1.191765, CPU=1.191765


In [10]:
# Check overall difference
max_error = np.max(np.abs(result_gpu - cpu_poly))
print(f"\nMax absolute error: {max_error:.2e}")
print(f"GPU execution time: {gpu_time:.4f} seconds")


Max absolute error: 1.22e-04
GPU execution time: 2.1780 seconds
